In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
import json
with open("/bohr/training-set-h31o/v1/train.json", "r") as f:
    train_ds = json.load(f)

train_ds = pd.DataFrame({
    "X": train_ds.keys(), "y": train_ds.values()
})

In [ ]:
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter

In [ ]:
def tokenize(word):
    return [w for w in word]
train_tokens = train_ds["X"].apply(tokenize)

def build_vocab(tokens):
    counter = Counter()
    for t in tokens:
        counter.update(t)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for w, c in counter.items():
        vocab[w] = len(vocab)
    return vocab

def build_seq(tokens, vocab):
    return [[vocab.get(t, 1) for t in w] for w in tokens]

vocab = build_vocab(train_tokens)
train_seq = build_seq(train_tokens, vocab)

In [ ]:
max_len = 35

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class DF(Dataset):
    def __init__(self, seqs, targets):
        self.seqs = seqs
        self.targets = targets
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        return self.seqs[idx], self.targets[idx]

def collate_fn(batch):
    x, y = zip(*batch)
    x = [torch.tensor(seq, dtype=torch.long) for seq in x]
    y = [torch.tensor(tar, dtype=torch.long) for tar in y]
    x = pad_sequence(x, batch_first=True, padding_value=vocab["<PAD>"])
    y = pad_sequence(y, batch_first=True, padding_value=-100)
    return x, y

train_df = DF(train_seq, list(train_ds["y"]))
train_dl = DataLoader(
    train_df, batch_size=16, shuffle=True,
    num_workers=4, pin_memory=True, collate_fn=collate_fn
)

for batch in train_dl:
    x, y = batch
    print(x.shape)
    print(y)
    break

In [ ]:
class MODEL(nn.Module):
    def __init__(self, vocab_len):
        super().__init__()
        self.embeds = nn.Embedding(vocab_len, 256, padding_idx=0)
        self.lstm = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True  , dropout =0.2)
        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.embeds(x)           
        out, _ = self.lstm(x)        
        return self.fc(out)         

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MODEL(len(vocab)).to(device)
model

In [ ]:
total_zero = 0
total_one = 0
for idx, row in train_ds.iterrows():
    for y in row["y"]:
        if y == 0:
            total_zero += 1
        else:
            total_one += 1
print(total_one)
print(total_zero)

In [ ]:
one_w  = total_zero / (total_one + total_zero)   
zero_w = total_one  / (total_one + total_zero)

In [ ]:
model = MODEL(len(vocab)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=2e-5)
loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100,
    weight=torch.tensor([zero_w, one_w], dtype=torch.float).to(device)
)

for e in range(70):
    t_loss = 0
    model.train()
    for batch in train_dl:
        x, y = batch
        x, y = x.to(device), y.to(device)
        out = model(x)                                          
        loss = loss_fn(out.reshape(-1, 2), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        t_loss += loss.item()
    torch.cuda.empty_cache()
    print(f"E: {e}  train_loss: {t_loss / len(train_dl):.4f}")
torch.save({"model_state": model.state_dict(), "vocab": vocab}, "model_checkpoint.pt")
print("Checkpoint saved.")

In [ ]:

import json
import torch
import torch.nn.functional as F
import os
import zipfile
import logging


def load_model(checkpoint_path: str, device: str = "cpu"):
    ckpt  = torch.load(checkpoint_path, map_location=device)
    vocab = ckpt["vocab"]
    m     = MODEL(len(vocab)).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    return m, vocab


def predict_word(word: str, model, vocab, device: str = "cpu") -> list:
    idxs = [vocab.get(c, 1) for c in word]          
    x    = torch.tensor(idxs, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)                           
    probs      = F.softmax(logits, dim=-1)
    boundaries = (probs[0, :len(word), 1].cpu() > 0.5).int().tolist()
    return boundaries
def predict_and_save(model, vocab, input_file: str, output_file: str, device: str = "cpu"):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    model.eval()
    predictions = {
        word: predict_word(word, model, vocab, device)
        for word in data
    }
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=4, ensure_ascii=False)
    logging.info(f"Predictions saved to {output_file}")

In [ ]:
if os.environ.get('ANSWER_PATH'):
    PATH = os.environ.get("ANSWER_PATH") + "/"
else:
    print("Test set not available during debugging — this is normal.")
    PATH = None

if PATH:
    predict_and_save(model, vocab, PATH + "val.json",  "submissionval.json",  device)
    predict_and_save(model, vocab, PATH + "test.json", "submissiontest.json", device)

In [ ]:
files_to_zip  = ['submissionval.json', 'submissiontest.json']
zip_filename  = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} created successfully!')

In [ ]:
# ── Standalone inference (no dataset, no training) ──────────────────────────
# Use this cell after training is done and checkpoint is saved.

# model, vocab = load_model("model_checkpoint.pt", device="cpu")
# print(predict_word("hello", model, vocab))
# print(predict_word("international", model, vocab))